# RDPro NLP To Scala Direct

Start from free-text NLP and generate Scala through the direct IR path.


In [1]:
from pathlib import Path
import json
import sys
import importlib

PROJECT_ROOT = Path('/Users/clockorangezoe/Documents/phd_projects/code/geoAI/LangGraph').resolve()
NOTEBOOK_DIR = PROJECT_ROOT / 'Notebook'
CODEGEN_DIR = NOTEBOOK_DIR / 'rdpro_section_codegen'
if str(NOTEBOOK_DIR) not in sys.path:
    sys.path.insert(0, str(NOTEBOOK_DIR))

for mod in [
    'rdpro_section_codegen',
    'rdpro_section_codegen.models',
    'rdpro_section_codegen.analyzer',
    'rdpro_section_codegen.planner',
    'rdpro_section_codegen.section_specs',
    'rdpro_section_codegen.langgraph_section_agent',
]:
    if mod in sys.modules:
        del sys.modules[mod]

import rdpro_section_codegen
import rdpro_section_codegen.models
import rdpro_section_codegen.analyzer
import rdpro_section_codegen.planner
import rdpro_section_codegen.section_specs
import rdpro_section_codegen.langgraph_section_agent

importlib.reload(rdpro_section_codegen.models)
importlib.reload(rdpro_section_codegen.analyzer)
importlib.reload(rdpro_section_codegen.planner)
importlib.reload(rdpro_section_codegen.section_specs)
importlib.reload(rdpro_section_codegen.langgraph_section_agent)
importlib.reload(rdpro_section_codegen)

from rdpro_section_codegen import analyze_python_script, build_plan
from rdpro_section_codegen.planner import map_apis_to_sections
from rdpro_section_codegen.section_specs import build_section_specs
from rdpro_section_codegen.langgraph_section_agent import _infer_free_text_task, _generate_python_reference


/Users/clockorangezoe/miniconda3/envs/geo_llm_spark/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
FREE_TEXT_TASK = """
Calculate land-use category percentages and summary statistics for Boston neighborhoods from NLCD raster data.
Input raster path: /Users/clockorangezoe/Downloads/Annual_NLCD_LndCov_2024_CU_C1V1/Annual_NLCD_LndCov_2024_CU_C_1V1_clip.tif.
Vector neighborhood polygon path: /Users/clockorangezoe/Downloads/boston_neighborhood_boundaries/Boston_Neighborhood_Boundaries_sample.shp.
This is a per-polygon zonal workflow.
Produce tabular CSV outputs with per-neighborhood class percentages and a neighborhood-level dominant class summary.
""".strip()

TRANSLATION_MODE = 'direct'
PROVIDER = 'openai'
MODEL = 'gpt-4o'

print(FREE_TEXT_TASK)
print('translation_mode =', TRANSLATION_MODE)


Calculate land-use category percentages and summary statistics for Boston neighborhoods from NLCD raster data.
Input raster path: /Users/clockorangezoe/Downloads/Annual_NLCD_LndCov_2024_CU_C1V1/Annual_NLCD_LndCov_2024_CU_C_1V1_clip.tif.
Vector neighborhood polygon path: /Users/clockorangezoe/Downloads/boston_neighborhood_boundaries/Boston_Neighborhood_Boundaries_sample.shp.
This is a per-polygon zonal workflow.
Produce tabular CSV outputs with per-neighborhood class percentages and a neighborhood-level dominant class summary.
translation_mode = direct


In [3]:
cfg = {
    'provider': PROVIDER,
    'model': MODEL,
}

task_info = _infer_free_text_task(FREE_TEXT_TASK, cfg)
print('LLM task info:', task_info)

python_reference = ''
if TRANSLATION_MODE in {'via-python', 'via-python-raw'}:
    python_reference = _generate_python_reference(FREE_TEXT_TASK, cfg)
    print('Generated Python reference:')
    print(python_reference)
    analysis = analyze_python_script(
        python_reference,
        task_type=task_info['task_type'],
        task_label=task_info.get('task_label', ''),
    )
else:
    analysis = analyze_python_script(
        FREE_TEXT_TASK,
        task_type=task_info['task_type'],
        task_label=task_info.get('task_label', ''),
    )

plan = build_plan(analysis)
print(json.dumps(plan.to_dict(), indent=2))


LLM task info: {'task_type': 'zonal_stats', 'task_label': 'Land-use category percentages and summary statistics for Boston neighborhoods', 'reason': 'The task involves calculating land-use category percentages and summary statistics for specific polygons (neighborhoods) using raster data, which aligns with zonal statistics operations.'}
{
  "task_type": "zonal_stats",
  "apis": [
    "geoTiff",
    "tile metadata",
    "pixel type",
    "shapefile",
    "raptorJoin",
    "csv output"
  ],
  "sections": [
    "LOAD_DATA",
    "TYPE_CHECK",
    "RASTER_VECTOR_JOIN",
    "ANALYTICS"
  ],
  "reasons": {
    "LOAD_DATA": "materialize configured raster/vector inputs from bound paths; selected APIs: geoTiff, shapefile",
    "TYPE_CHECK": "validate pixel/data assumptions introduced during load; selected APIs: tile metadata, pixel type",
    "RASTER_VECTOR_JOIN": "materialize the explicit raster-vector join as joined_records; selected APIs: raptorJoin",
    "ANALYTICS": "derive per_feature_tabu

## Reasoning IR

Inspect the reasoning-centered intermediate representation induced from NLP or the generated Python scaffold.

In [4]:
print('Workflow goal:', analysis.workflow_goal)
print('Output intent:', analysis.output_intent)
print('Output required input shape:', analysis.output_required_input_shape)
print('Output write strategy:', analysis.output_write_strategy)
print('Scalability constraints:')
for item in analysis.scalability_constraints:
    print('  -', item)
print()
print('Section goals:')
for key, value in analysis.section_goals.items():
    print(f'  {key}: {value}')
print()
print('Intermediate contracts:')
for key, value in analysis.intermediate_contracts.items():
    print(f'  {key}: {value}')
print()
print('Dataflow lineage:')
for step in analysis.dataflow_lineage:
    print('  -', step)
print()
print('Full semantic_ir:')
print(json.dumps(analysis.semantic_ir, indent=2))


Workflow goal: Land-use category percentages and summary statistics for Boston neighborhoods using raster and vector inputs
Output intent: write distributed tabular output as csv with persisted_file semantics
Output required input shape: flat_tabular_rows
Output write strategy: distributed_csv
Scalability constraints:
  - avoid collect() and other driver-heavy materialization
  - avoid validation-only count() on large datasets
  - preserve reusable distributed intermediates across sections
  - prefer distributed raster-vector operations over local geometry loops
  - prefer distributed, overwrite-safe output behavior

Section goals:
  LOAD_DATA: materialize configured raster/vector inputs from bound paths
  TYPE_CHECK: validate pixel/data assumptions introduced during load
  SPATIAL_CHECK: verify CRS/alignment requirements only when spatial compatibility matters
  TRANSFORM: prepare raster inputs for the explicit raster_vector_join join stage
  ANALYTICS: derive per_feature_tabular_summ

## API To Section Map

In [5]:
section_map = map_apis_to_sections(analysis, plan.apis)
print(json.dumps(section_map, indent=2))


{
  "LOAD_DATA": [
    "geoTiff",
    "shapefile"
  ],
  "TYPE_CHECK": [
    "tile metadata",
    "pixel type"
  ],
  "SPATIAL_CHECK": [],
  "TRANSFORM": [],
  "RASTER_VECTOR_JOIN": [
    "raptorJoin"
  ],
  "ANALYTICS": []
}


## Section Specs

In [6]:
section_specs = build_section_specs(
    analysis,
    plan,
    (CODEGEN_DIR / 'job_scaffold.scala').read_text(encoding='utf-8', errors='ignore'),
)
print(json.dumps(section_specs, indent=2))


{
  "task_type": "zonal_stats",
  "task_label": "Land-use category percentages and summary statistics for Boston neighborhoods",
  "apis": [
    "geoTiff",
    "tile metadata",
    "pixel type",
    "shapefile",
    "raptorJoin",
    "csv output"
  ],
  "sections": [
    "LOAD_DATA",
    "TYPE_CHECK",
    "RASTER_VECTOR_JOIN",
    "ANALYTICS"
  ],
  "section_api_map": {
    "LOAD_DATA": [
      "geoTiff",
      "shapefile"
    ],
    "TYPE_CHECK": [
      "tile metadata",
      "pixel type"
    ],
    "SPATIAL_CHECK": [],
    "TRANSFORM": [],
    "RASTER_VECTOR_JOIN": [
      "raptorJoin"
    ],
    "ANALYTICS": []
  },
  "inputs": {
    "raster_inputs": [
      "/Users/clockorangezoe/Downloads/Annual_NLCD_LndCov_2024_CU_C1V1/Annual_NLCD_LndCov_2024_CU_C_1V1_clip.tif"
    ],
    "vector_inputs": [
      "/Users/clockorangezoe/Downloads/boston_neighborhood_boundaries/Boston_Neighborhood_Boundaries_sample.shp"
    ],
    "output_candidates": [],
    "requires_vector": true,
    "multi_ra

In [7]:
print('Task type:', plan.task_type)
print('Task label:', plan.analysis.get('task_label', ''))
print('Capabilities:', plan.analysis.get('capabilities', []))
print('APIs:', plan.apis)
print('Sections:', plan.sections)
print('Reasons:')
for key, value in plan.reasons.items():
    print(f'  {key}: {value}')


Task type: zonal_stats
Task label: Land-use category percentages and summary statistics for Boston neighborhoods
Capabilities: ['vector_input', 'raster_vector_summary', 'tabular_output']
APIs: ['geoTiff', 'tile metadata', 'pixel type', 'shapefile', 'raptorJoin', 'csv output']
Sections: ['LOAD_DATA', 'TYPE_CHECK', 'RASTER_VECTOR_JOIN', 'ANALYTICS']
Reasons:
  LOAD_DATA: materialize configured raster/vector inputs from bound paths; selected APIs: geoTiff, shapefile
  TYPE_CHECK: validate pixel/data assumptions introduced during load; selected APIs: tile metadata, pixel type
  RASTER_VECTOR_JOIN: materialize the explicit raster-vector join as joined_records; selected APIs: raptorJoin
  ANALYTICS: derive per_feature_tabular_summary from joined_records


## Run Section Agent

Launch the section-by-section generate/compile/fix loop from the free-text task. The generated Scala is written to a separate output file, not the scaffold.

In [8]:
import json
import subprocess

ENV_PYTHON = Path('/Users/clockorangezoe/miniconda3/envs/geo_llm_spark/bin/python')
SECTION_AGENT = CODEGEN_DIR / 'langgraph_section_agent.py'
SCAFFOLD_SCALA = CODEGEN_DIR / 'job_scaffold.scala'
OUTPUT_SECTIONAL = CODEGEN_DIR / f'nlp_to_scala_{TRANSLATION_MODE}.scala'
WORK_DIR = CODEGEN_DIR / 'generated_output' / f'nlp_to_scala_{TRANSLATION_MODE}'
API_DOC = PROJECT_ROOT / 'Doc' / 'rdpro_api_doc_combined.md'
GUIDE = NOTEBOOK_DIR / 'RDProAgentLoop_perAPI_fix_migration_guide.md'

WORK_DIR.mkdir(parents=True, exist_ok=True)

cmd = [
    str(ENV_PYTHON),
    str(SECTION_AGENT),
    '--translation-mode', TRANSLATION_MODE,
    '--free-text', FREE_TEXT_TASK,
    '--scaffold', str(SCAFFOLD_SCALA),
    '--output-scala', str(OUTPUT_SECTIONAL),
    '--work-dir', str(WORK_DIR),
    '--api-doc', str(API_DOC),
    '--guide', str(GUIDE),
    '--provider', PROVIDER,
    '--model', MODEL,
]

print('Running command:')
print(' '.join(cmd))
result = subprocess.run(cmd, cwd=str(CODEGEN_DIR), capture_output=True, text=True)
print('returncode =', result.returncode)

summary = {}
stdout_text = (result.stdout or '').strip()
if stdout_text:
    match = __import__('re').search(r'\{.*\}', stdout_text, flags=__import__('re').S)
    if match:
        try:
            summary = json.loads(match.group(0))
        except Exception:
            print('stdout contained JSON-like text but could not be parsed')
            print(stdout_text[:2000])
    else:
        print('stdout did not contain JSON')
        print(stdout_text[:2000])

if summary:
    print(json.dumps({
        'success': summary.get('success'),
        'fail_reason': summary.get('fail_reason'),
        'planned_sections': summary.get('planned_sections'),
        'completed_sections': summary.get('completed_sections'),
        'output_scala': summary.get('output_scala'),
        'last_report': summary.get('last_report'),
        'last_merged': summary.get('last_merged'),
    }, indent=2))

if result.stderr:
    print('----- stderr -----')
    print(result.stderr[:4000])

print('Scala output exists =', OUTPUT_SECTIONAL.exists())




Running command:
/Users/clockorangezoe/miniconda3/envs/geo_llm_spark/bin/python /Users/clockorangezoe/Documents/phd_projects/code/geoAI/LangGraph/Notebook/rdpro_section_codegen/langgraph_section_agent.py --translation-mode direct --free-text Calculate land-use category percentages and summary statistics for Boston neighborhoods from NLCD raster data.
Input raster path: /Users/clockorangezoe/Downloads/Annual_NLCD_LndCov_2024_CU_C1V1/Annual_NLCD_LndCov_2024_CU_C_1V1_clip.tif.
Vector neighborhood polygon path: /Users/clockorangezoe/Downloads/boston_neighborhood_boundaries/Boston_Neighborhood_Boundaries_sample.shp.
This is a per-polygon zonal workflow.
Produce tabular CSV outputs with per-neighborhood class percentages and a neighborhood-level dominant class summary. --scaffold /Users/clockorangezoe/Documents/phd_projects/code/geoAI/LangGraph/Notebook/rdpro_section_codegen/job_scaffold.scala --output-scala /Users/clockorangezoe/Documents/phd_projects/code/geoAI/LangGraph/Notebook/rdpro_sec